# 01 — Analisis Data Eksploratif (EDA)

Notebook ini memeriksa snapshot utama IndoDiscourse yang dipin tanpa menampilkan teks, ID, contoh baris, n-gram, atau isi token. Tidak ada teks, text_id, contoh baris, n-gram, atau isi token yang ditampilkan. Diagnostik skema hanya mencatat indeks baris dan kolom; grafik serta ringkasan lain hanya memuat agregat. Output Kaggle tetap privat karena agregat pun dapat memberi informasi tentang data.

Sumber yang diunduh hanya berkas JSONL kanonis pada revisi 8af8c0f00b05633c05cc41fbd7650178ababc892. Cache berada di /tmp, diverifikasi dengan SHA-256, dan tidak disalin ke /kaggle/working. Konfigurasi annotator dan demografi tidak dimuat. EDA tidak mengubah teks atau membentuk sampel penelitian.

Bagian berikut merangkum provenance dan skema, suara serta label, duplikat dan prevalensi, penanda teks, panjang, dan tokenizer. Tiap bagian menjelaskan implikasi metodologi. Analisis setelah validasi memakai subset eksploratif baris valid; subset ini bukan sampel penelitian.

In [ ]:
import hashlib, importlib.metadata, json, os, platform, re, tempfile, unicodedata, urllib.request
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO = "Exqrch/IndoDiscourse"
REVISION = "8af8c0f00b05633c05cc41fbd7650178ababc892"
FILENAME = "indotoxic2024_annotated_data_v2_final.jsonl"
EXPECTED_SHA = "94bb43546f5040a606a8d842694b7d490df043051377a6580e1e95ad2c7334e7"
EXPECTED_ROWS = 28448
TOKENIZER_REPO = "indolem/indobertweet-base-uncased"
TOKENIZER_REVISION = "32e28c05b47e33b6675d2670a1745c50a65e987a"
MAX_LENGTH = 512
COLUMNS = ["text_id", "annotators_id", "text", "initial_paragraph", "topic",
           "is_noise_or_spam_text", "related_to_election_2024", "toxicity", "polarized",
           "profanity_obscenity", "threat_incitement_to_violence", "insults",
           "identity_attack", "sexually_explicit"]
STRINGS = {"text_id", "text", "initial_paragraph", "topic"}
LISTS = set(COLUMNS) - STRINGS
NULLABLE = {"initial_paragraph", "topic", "is_noise_or_spam_text", "related_to_election_2024",
            "profanity_obscenity", "threat_incitement_to_violence", "insults",
            "identity_attack", "sexually_explicit"}
OUT = Path("/kaggle/working/eda")
OUT.mkdir(parents=True, exist_ok=True)
for stale in ("eda_report.json", "label_pairs.png", "text_lengths.png", "text_markers.png", "subword_lengths.png"):
    (OUT / stale).unlink(missing_ok=True)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def safe_n(value):
    return "<10" if int(value) < 10 else int(value)

def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

url = f"https://huggingface.co/datasets/{REPO}/resolve/{REVISION}/{FILENAME}"
cache_dir = Path("/tmp/indodiscourse_eda")
cache_dir.mkdir(parents=True, exist_ok=True)
cache = cache_dir / f"{REVISION}_{FILENAME}"
if not cache.exists():
    fd, name = tempfile.mkstemp(prefix="download-", suffix=".jsonl", dir=cache_dir)
    os.close(fd); tmp = Path(name)
    try:
        with urllib.request.urlopen(url, timeout=120) as response, tmp.open("wb") as target:
            while chunk := response.read(1024 * 1024):
                target.write(chunk)
        if sha256(tmp) != EXPECTED_SHA:
            raise ValueError("SHA-256 berkas sumber tidak cocok.")
        os.replace(tmp, cache)
    finally:
        tmp.unlink(missing_ok=True)
raw_sha = sha256(cache)
if raw_sha != EXPECTED_SHA:
    raise ValueError("SHA-256 cache tidak cocok dengan revisi yang dipin.")

records, parse_issues = [], []
with cache.open(encoding="utf-8") as stream:
    for index, line in enumerate(stream):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            records.append(None); parse_issues.append((index, "<json>"))

def validate(row):
    problems = []
    if not isinstance(row, dict):
        return ["<row>"]
    if set(row) != set(COLUMNS):
        problems.append("<columns>")
    for col in COLUMNS:
        if col not in row:
            problems.append(col); continue
        value = row[col]
        if value is None and col in NULLABLE:
            continue
        if col in STRINGS and not isinstance(value, str):
            problems.append(col)
        if col in LISTS and not (isinstance(value, list) and all(isinstance(x, str) for x in value)):
            problems.append(col)
    ids = row.get("annotators_id")
    if isinstance(row.get("text_id"), str) and not row["text_id"].strip():
        problems.append("text_id")
    valid_ids = isinstance(ids, list) and bool(ids) and all(
        isinstance(value, str) and value.strip() for value in ids)
    if isinstance(ids, list) and not valid_ids:
        problems.append("annotators_id")
    for col in ("toxicity", "polarized"):
        votes = row.get(col)
        if isinstance(votes, list):
            valid_votes = bool(votes) and all(
                isinstance(value, str) and value in {"0", "1"} for value in votes)
            if not valid_votes:
                problems.append(col)
            if isinstance(ids, list) and len(votes) != len(ids):
                problems.append(col)
    return sorted(set(problems))

anomalies, valid = list(parse_issues), []
for index, row in enumerate(records):
    problems = validate(row)
    if problems:
        anomalies.extend((index, col) for col in problems)
    else:
        valid.append((index, row))
if len(records) != EXPECTED_ROWS:
    anomalies.append((None, "<row_count>"))
anomalies = sorted(set(anomalies), key=lambda item: (-1 if item[0] is None else item[0], item[1]))
versions = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
            "matplotlib": version("matplotlib"), "transformers": version("transformers"),
            "tokenizers": version("tokenizers")}
print("Sumber:", REPO, REVISION, "| SHA-256:", raw_sha)
print("Baris aktual:", len(records), "| lolos validasi:", len(valid))

## 1. Provenance dan validasi skema

Validasi memeriksa kolom, tipe dasar, suara biner untuk kedua tugas utama, dan panjang daftar suara dibanding jumlah anotator. Anomali hanya ditampilkan sebagai indeks baris dan nama kolom. Nilai sumber tidak ditampilkan.

**Implikasi untuk metodologi.** Anomali perlu ditangani melalui perubahan metode yang disetujui. Notebook ini tidak memperbaiki atau menghapusnya diam-diam. Baris yang tidak lolos validasi tidak masuk ke subset analisis eksploratif.

In [ ]:
display(pd.DataFrame({
    "kolom": COLUMNS,
    "tipe": ["string" if col in STRINGS else "list[string]" for col in COLUMNS],
    "null": [safe_n(sum(row.get(col) is None for row in records if isinstance(row, dict))) for col in COLUMNS],
}))
if anomalies:
    display(pd.DataFrame(anomalies, columns=["indeks_baris", "kolom"]))
else:
    print("Tidak ditemukan anomali skema.")

## 2. Suara, kesepakatan, dan label

Strict majority diterapkan terpisah pada toxicity dan polarization. Seri pada salah satu tugas mengecualikan baris dari hitungan eksploratif. Hanya string kosong persis dengan pasangan label (0, 0) dikecualikan; whitespace-only tetap dipertahankan. Kesepakatan adalah rerata kesepakatan antar pasangan anotator per baris.

Pasangan label aktual dibandingkan dengan angka proposal 20.436, 2.591, 933, dan 895 (total 24.855). Selisih aktual dikurangi angka proposal ini hanya pemeriksaan: subset valid bukan sampel final. Korelasi bersifat deskriptif dan tidak membuktikan manfaat multi-task learning.

**Implikasi untuk metodologi.** Seri dan distribusi label diaudit sesuai proposal; hasil ini tidak mengubah sampel, menyeimbangkan kelas, atau menyimpulkan keunggulan model.

In [ ]:
def majority(votes):
    ones = votes.count("1")
    return None if 2 * ones == len(votes) else int(2 * ones > len(votes))

def agreement(votes):
    n, zeros, ones = len(votes), votes.count("0"), votes.count("1")
    return None if n < 2 else (zeros * (zeros - 1) + ones * (ones - 1)) / (n * (n - 1))

annotator_hist = Counter(len(row["annotators_id"]) for _, row in valid)
display(pd.DataFrame([{"anotator_per_baris": n, "jumlah_baris": safe_n(count)}
                      for n, count in sorted(annotator_hist.items())]))
vote_summary = []
for task in ("toxicity", "polarized"):
    votes = [row[task] for _, row in valid]
    flat = [vote for row_votes in votes for vote in row_votes]
    agreements = [agreement(row_votes) for row_votes in votes]
    agreements = [value for value in agreements if value is not None]
    vote_summary.append({"tugas": task, "suara_0": safe_n(flat.count("0")),
                         "suara_1": safe_n(flat.count("1")),
                         "baris_seri": safe_n(sum(majority(value) is None for value in votes)),
                         "kesepakatan_pasangan": round(float(np.mean(agreements)), 4) if agreements else None})
display(pd.DataFrame(vote_summary))

exploratory = []
for row_id, row in valid:
    toxicity, polarized = majority(row["toxicity"]), majority(row["polarized"])
    if toxicity is None or polarized is None:
        continue
    if row["text"] == "" and toxicity == 0 and polarized == 0:
        continue
    exploratory.append({"row_id": row_id, "text": row["text"],
                        "toxicity_label": toxicity, "polarized_label": polarized})
eda_df = pd.DataFrame(exploratory)
reference = {(0, 0): 20436, (0, 1): 2591, (1, 0): 933, (1, 1): 895}
actual = Counter(zip(eda_df.get("toxicity_label", []), eda_df.get("polarized_label", [])))
display(pd.DataFrame([{"tugas": task, "label": label,
                       "jumlah": safe_n((eda_df[col] == label).sum()) if len(eda_df) else "<10"}
                      for task, col in (("toxicity", "toxicity_label"), ("polarized", "polarized_label"))
                      for label in (0, 1)]))
marginal_reference = {("toxicity", 0): 23027, ("toxicity", 1): 1828,
                     ("polarized", 0): 21369, ("polarized", 1): 3486}
marginal_rows = []
for task, col in (("toxicity", "toxicity_label"), ("polarized", "polarized_label")):
    for label in (0, 1):
        count = int((eda_df[col] == label).sum()) if len(eda_df) else 0
        expected = marginal_reference[(task, label)]
        marginal_rows.append({"tugas": task, "label": label, "aktual": safe_n(count),
                              "proposal": expected,
                              "aktual_minus_proposal": safe_n(count) if count < 10 else count - expected})
display(pd.DataFrame(marginal_rows))
joint = []
for pair, expected in reference.items():
    count = int(actual.get(pair, 0))
    joint.append({"pasangan": f"{pair[0]}-{pair[1]}", "aktual_subset_valid": safe_n(count),
                  "proposal": expected, "aktual_minus_proposal": safe_n(count) if count < 10 else count - expected})
display(pd.DataFrame(joint))
visible = [(pair, int(actual.get(pair, 0))) for pair in reference if actual.get(pair, 0) >= 10]
if visible:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar([f"{a}-{b}" for (a, b), _ in visible], [count for _, count in visible])
    ax.set(title="Pasangan label pada subset eksploratif", xlabel="Toxicity–polarized",
           ylabel="Baris; kategori <10 disembunyikan")
    fig.tight_layout(); fig.savefig(OUT / "label_pairs.png", dpi=150); plt.show(); plt.close(fig)
if len(eda_df) and all(actual.get(pair, 0) >= 10 for pair in reference):
    print("Korelasi Pearson label biner (deskriptif):",
          round(float(np.corrcoef(eda_df["toxicity_label"], eda_df["polarized_label"])[0, 1]), 4))
else:
    print("Korelasi tidak ditampilkan karena ada agregat pasangan di bawah 10.")

## 3. Duplikat, noise/spam, topik, dan label rinci

Kunci kelompok memakai NFKC, casefold, trim, dan penggabungan whitespace. Nilai kunci tidak ditampilkan; baris duplikat dan noise/spam tetap dipertahankan. Topik dan kolom label rinci diringkas sebagai prevalensi agregat.

**Implikasi untuk metodologi.** Duplikat mendukung group key untuk split mendatang, tetapi notebook ini tidak membuat fold. Konflik label menjadi temuan deskriptif, bukan dasar menghapus baris.

In [ ]:
def make_group_key(text):
    return " ".join(unicodedata.normalize("NFKC", text).casefold().split())

if len(eda_df):
    eda_df["group_key"] = eda_df["text"].map(make_group_key)
    sizes = eda_df.groupby("group_key", sort=False).size()
    duplicate_sizes = sizes[sizes > 1]
    unique_labels = eda_df.groupby("group_key")[["toxicity_label", "polarized_label"]].nunique()
    conflicts = unique_labels.gt(1).any(axis=1)
    display(pd.DataFrame([{"group_size": size, "groups": safe_n(count)}
                          for size, count in sorted(duplicate_sizes.value_counts().items())]))
    print("Kelompok duplikat:", safe_n(len(duplicate_sizes)),
          "| konflik label:", safe_n(int(conflicts[duplicate_sizes.index].sum())))

prevalence = []
for col in ("is_noise_or_spam_text", "related_to_election_2024", "profanity_obscenity",
            "threat_incitement_to_violence", "insults", "identity_attack", "sexually_explicit"):
    labels = [majority(row[col]) for _, row in valid if isinstance(row.get(col), list) and row[col]
              and all(value in {"0", "1"} for value in row[col])]
    labels = [value for value in labels if value is not None]
    positives = sum(labels)
    prevalence.append({"kolom": col, "baris_tidak_seri": safe_n(len(labels)),
                       "label_1": safe_n(positives),
                       "persen_label_1": round(100 * positives / len(labels), 2)
                       if len(labels) >= 10 and positives >= 10 else "tersensor"})
display(pd.DataFrame(prevalence))
topic_counts = Counter(row["topic"] for _, row in valid
                       if isinstance(row.get("topic"), str) and row["topic"] != "")
display(pd.DataFrame([{"topik": topic, "jumlah": safe_n(count)}
                      for topic, count in topic_counts.most_common(20)]))

## 4. Panjang dan penanda teks

Semua ukuran berasal dari teks asli. Detektor emoji memakai rentang Unicode pictograph/dingbat umum sebagai pendekatan. URL, mention, dan hashtag memakai pola karakter sederhana. Karakter kontrol dihitung dari kategori Unicode Cc; isi teks tidak ditampilkan.

**Implikasi untuk metodologi.** Prevalensi memberi konteks tentang materi yang mungkin ditemui model. Proposal tetap dipatuhi: tidak menghapus atau menormalisasi emoji, slang, URL, tanda baca, profanity, atau bahasa campuran secara manual.

In [ ]:
URL = re.compile(r"(?:https?://|www\.)[^\s]+", re.I)
MENTION = re.compile(r"(?<!\w)@[\w]+")
HASHTAG = re.compile(r"(?<!\w)#[\w]+")
EMOJI = re.compile(r"[\U0001F000-\U0001FAFF\u2600-\u27BF]")

def markers(text):
    letters = [c for c in text if c.isalpha()]
    upper, lower = any(c.isupper() for c in letters), any(c.islower() for c in letters)
    return {"emoji_like": bool(EMOJI.search(text)), "url": bool(URL.search(text)),
            "mention": bool(MENTION.search(text)), "hashtag": bool(HASHTAG.search(text)),
            "uppercase": upper, "lowercase": lower, "mixed_case": upper and lower,
            "non_ascii": any(ord(c) > 127 for c in text),
            "nfkc_would_change": unicodedata.normalize("NFKC", text) != text,
            "control_character": any(unicodedata.category(c) == "Cc" for c in text),
            "whitespace_only": text != "" and not text.strip()}

texts = eda_df["text"].tolist() if len(eda_df) else []
mark = [markers(text) for text in texts]
marker_names = ("emoji_like", "url", "mention", "hashtag", "uppercase", "lowercase",
                "mixed_case", "non_ascii", "nfkc_would_change", "control_character", "whitespace_only")
marker_rows = []
for name in marker_names:
    count = sum(item[name] for item in mark)
    marker_rows.append({"penanda": name, "baris": safe_n(count),
                        "persen": round(100 * count / len(texts), 2)
                        if count >= 10 and len(texts) >= 10 else "tersensor"})
display(pd.DataFrame(marker_rows))
lengths = pd.DataFrame({"karakter": [len(t) for t in texts],
                        "kata_whitespace": [len(t.split()) for t in texts]})
if len(lengths):
    display(lengths.quantile([0, .25, .5, .75, .9, .95, .99, 1]).round(2))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(lengths["karakter"], bins=40, density=True)
    axes[0].set(title="Panjang karakter", ylabel="Kepadatan")
    axes[1].hist(lengths["kata_whitespace"], bins=40, density=True)
    axes[1].set(title="Kata berbasis whitespace")
    fig.tight_layout(); fig.savefig(OUT / "text_lengths.png", dpi=150); plt.show(); plt.close(fig)
visible = [(row["penanda"], row["persen"]) for row in marker_rows
           if isinstance(row["persen"], (int, float))]
if visible:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar([name for name, _ in visible], [pct for _, pct in visible])
    ax.set(title="Prevalensi penanda teks; agregat ≥10", ylabel="Persen baris")
    ax.tick_params(axis="x", rotation=45); fig.tight_layout()
    fig.savefig(OUT / "text_markers.png", dpi=150); plt.show(); plt.close(fig)

## 5. Tokenizer dan batas panjang

Tokenizer indolem/indobertweet-base-uncased dipakai pada revisi tetap; hanya tokenizer dimuat, tanpa bobot model, pada CPU. Panjang mencakup special tokens. Fertility adalah subword non-special per kata whitespace. Token ID dan isi token tidak ditampilkan atau disimpan.

Proposal mencatat sekitar 98,45% teks di bawah batas 512 dan 386 teks mencapai batas. Angka tersebut hanya pembanding, bukan target lulus; subset ini bukan sampel final.

**Implikasi untuk metodologi.** Distribusi membantu memeriksa relevansi batas 512 dan truncation yang ditetapkan proposal. EDA tidak mengubah strategi tokenisasi.

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_REPO, revision=TOKENIZER_REVISION, use_fast=True)
subword_lengths, fertility = [], []
unknown, total_subwords = 0, 0
special = tokenizer.num_special_tokens_to_add(pair=False)
for start in range(0, len(texts), 128):
    batch = texts[start:start + 128]
    encoded = tokenizer(batch, add_special_tokens=True, truncation=False, padding=False)
    for text, ids in zip(batch, encoded["input_ids"]):
        content = max(0, len(ids) - special)
        subword_lengths.append(len(ids)); total_subwords += content
        if tokenizer.unk_token_id is not None:
            unknown += sum(token_id == tokenizer.unk_token_id for token_id in ids)
        words = len(text.split())
        if words:
            fertility.append(content / words)
if subword_lengths:
    arr = np.asarray(subword_lengths)
    below, reaches = int((arr < MAX_LENGTH).sum()), int((arr >= MAX_LENGTH).sum())
    tokenizer_report = {
        "repository": TOKENIZER_REPO, "revision": TOKENIZER_REVISION,
        "tokenizer_only": True, "device": "CPU", "text_count": len(arr),
        "length_quantiles": {str(q): float(np.quantile(arr, q)) for q in (0, .5, .9, .95, .99, 1)},
        "below_512_count": safe_n(below), "below_512_percent": round(100 * below / len(arr), 2),
        "at_or_above_512": safe_n(reaches), "above_512": safe_n(int((arr > 512).sum())),
        "mean_fertility": round(float(np.mean(fertility)), 4) if fertility else None,
        "median_fertility": round(float(np.median(fertility)), 4) if fertility else None,
        "fertility_texts": safe_n(len(fertility)),
        "unknown_occurrences": safe_n(unknown),
        "unknown_rate": round(unknown / total_subwords, 8) if unknown >= 10 and total_subwords else "<10",
        "proposal_checks_only": {"below_512_percent": 98.45, "texts_reaching_512": 386},
    }
    display(pd.DataFrame([
        {"ukuran": "di bawah 512 (%)", "aktual": tokenizer_report["below_512_percent"], "proposal": 98.45},
        {"ukuran": "mencapai atau melewati 512", "aktual": tokenizer_report["at_or_above_512"], "proposal": 386},
        {"ukuran": "fertility rata-rata", "aktual": tokenizer_report["mean_fertility"], "proposal": "—"},
        {"ukuran": "laju unknown", "aktual": tokenizer_report["unknown_rate"], "proposal": "—"},
    ]))
    fig, ax = plt.subplots(figsize=(9, 4)); ax.hist(arr, bins=40, density=True)
    ax.axvline(512, label="batas proposal: 512 token")
    ax.set(title="Panjang subword", xlabel="Subword termasuk special tokens", ylabel="Kepadatan")
    ax.legend(); fig.tight_layout(); fig.savefig(OUT / "subword_lengths.png", dpi=150)
    plt.show(); plt.close(fig)
else:
    tokenizer_report = {"repository": TOKENIZER_REPO, "revision": TOKENIZER_REVISION, "text_count": 0}
    print("Tidak ada teks dalam subset eksploratif.")

report = {
    "status": "complete_with_anomalies" if anomalies else "complete",
    "source": {"repository": REPO, "configuration": "main", "revision": REVISION,
               "filename": FILENAME, "sha256": raw_sha, "expected_rows": EXPECTED_ROWS,
               "actual_rows": len(records)},
    "valid_rows": len(valid), "anomalies": [{"row_index": i, "column": c} for i, c in anomalies],
    "exploratory_subset_rows": len(eda_df),
    "joint_counts_masked_below_10": {f"{a},{b}": safe_n(actual.get((a, b), 0)) for a, b in reference},
    "tokenizer": tokenizer_report, "versions": versions,
    "saved_output_scope": "aggregate reports and figures only",
}
write_json(OUT / "eda_report.json", report)
print("Laporan agregat dan gambar tersimpan di /kaggle/working/eda.")